# 02 – Background Studies

This notebook investigates the **composition and properties of the background** in the semitauonic B decay analysis.

**Goals:**
- Break down background contributions by decay mode
- Compare key discriminating variable distributions for each background component
- Estimate signal-to-background ratios before any BDT selection
- Identify the dominant background sources and their properties

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot

try:
    import mplhep as hep
    hep.style.use(hep.style.Belle2)
except ImportError:
    pass

plt.rcParams['figure.dpi'] = 120

## 1. Load Data

In [ ]:
# ─── Paths – update as needed ────────────────────────────────────────────────
SIGNAL_FILE   = "/path/to/signal_mc.root"
NORM_FILE     = "/path/to/normalization_mc.root"   # B → D(*) l ν_l
GENERIC_BB_FILE = "/path/to/generic_BB_mc.root"
CONTINUUM_FILE  = "/path/to/continuum_mc.root"
TREE_NAME       = "ntuple"
# ─────────────────────────────────────────────────────────────────────────────

def load(path, label):
    with uproot.open(path) as f:
        df = f[TREE_NAME].arrays(library="pd")
    df["source"] = label
    return df

sig      = load(SIGNAL_FILE,    "Signal")
norm     = load(NORM_FILE,      "Normalization")
gen_BB   = load(GENERIC_BB_FILE, "Generic BB")
cont     = load(CONTINUUM_FILE,  "Continuum")

all_samples = pd.concat([sig, norm, gen_BB, cont], ignore_index=True)
print(all_samples.groupby("source").size())

## 2. Background Composition

In [ ]:
# Event counts (before weighting) – update with cross-section weights as appropriate
counts = all_samples.groupby("source").size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(counts.index, counts.values, color="steelblue")
ax.set_xlabel("Number of events")
ax.set_title("Sample composition (unweighted)")
plt.tight_layout()
plt.show()

## 3. Key Variable Distributions by Source

In [ ]:
VARIABLES = ["M2_miss", "q2", "E_miss", "p_D_cms"]
COLORS    = {"Signal": "steelblue", "Normalization": "green",
             "Generic BB": "tomato",  "Continuum": "orange"}

for var in VARIABLES:
    if var not in all_samples.columns:
        print(f"Variable '{var}' not found – skipping.")
        continue

    fig, ax = plt.subplots(figsize=(6, 4))
    lo, hi = all_samples[var].quantile([0.01, 0.99])
    bins = np.linspace(lo, hi, 60)

    for source, grp in all_samples.groupby("source"):
        ax.hist(grp[var], bins=bins, density=True, histtype="step",
                linewidth=1.8, label=source, color=COLORS.get(source))

    ax.set_xlabel(var)
    ax.set_ylabel("Normalized counts")
    ax.legend()
    ax.set_title(f"Distribution of {var} by sample")
    plt.tight_layout()
    plt.show()

## 4. Signal-to-Background Ratio

In [ ]:
n_sig = len(sig)
n_bkg = len(all_samples[all_samples["source"] != "Signal"])

print(f"Signal events     : {n_sig:,}")
print(f"Background events : {n_bkg:,}")
print(f"S / (S+B)         : {n_sig / (n_sig + n_bkg):.4f}")
print(f"S / sqrt(B)       : {n_sig / np.sqrt(n_bkg):.2f}")

## 5. Notes and Conclusions

*(Fill in your findings here after running the notebook on real data.)*

- Dominant background: ...
- Best discriminating variables: ...
- Variables to include in BDT training: ...